<p class="lead">Tree ensembles are the working default for tabular scientific data: they need no scaling, tolerate irrelevant columns, capture interactions without being told about them, and still outperform deep networks on medium-sized tables<sup class="c"><a href="#r16">[16]</a></sup><sup class="c"><a href="#r17">[17]</a></sup>. This unit builds them from their two generating ideas — <strong>bagging</strong>, which averages away variance, and <strong>boosting</strong>, which fits the residuals of what it has learned so far — traces a gradient-boosting fit by hand on six points, and then spends most of its length on the activity that actually consumes a project's compute: hyperparameter search. Grid, random and a Bayesian-style surrogate search are run under one fixed budget on the same scaffold split, the result is wrapped in nested cross-validation so that the number reported is not the number that was optimised, and the unit ends by writing out the Methods paragraph that the search justifies. Along the way: learning and validation curves as diagnostics, a bias–variance decomposition by simulation, what class weighting really does under imbalance, and how to impose a monotone relationship when the chemistry demands one.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad construye los conjuntos de árboles desde sus dos ideas generadoras: el <em>bagging</em>, que promedia modelos de alta varianza (bosque aleatorio, con su estimación <em>out-of-bag</em>), y el <em>boosting</em>, que ajusta sucesivamente los residuos (trazado a mano sobre seis puntos y reproducido con código y con xgboost). Se comparan las implementaciones disponibles en R (ranger, randomForest, xgboost con crecimiento por profundidad y por hoja) y, sobre todo, se estudia la optimización de hiperparámetros: búsqueda en grilla, aleatoria y bayesiana con un modelo sustituto y poda por mediana, bajo un mismo presupuesto de 20 configuraciones y con validación cruzada anidada para obtener una estimación honesta. Se incluyen curvas de aprendizaje y de validación, la descomposición sesgo-varianza por simulación, el tratamiento del desbalance de clases mediante pesos y umbrales, y las restricciones de monotonía impuestas por el conocimiento químico. Termina con el párrafo de Métodos que corresponde a la búsqueda realmente ejecutada.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Explain bagging, random forests (including the out-of-bag estimate) and gradient boosting, and trace one boosting fit by hand.</li>
<li>Name the main hyperparameters of a boosted ensemble — learning rate, number of iterations, tree size, subsampling, regularisation — and say what each controls.</li>
<li>Compare the ensemble implementations available in R and know when the difference between them matters.</li>
<li>Tune hyperparameters with grid, random and Bayesian-style search under a stated budget, with pruning and fixed seeds, and report the search space and the chosen configuration.</li>
<li>Use nested cross-validation so that the reported score is not the score that was optimised.</li>
<li>Read learning and validation curves to decide between more data, more capacity and more regularisation, and demonstrate the bias–variance trade-off by simulation.</li>
<li>Handle class imbalance with weights and thresholds, and know what each does to discrimination and to calibration.</li>
<li>Impose monotonic constraints when domain knowledge requires them, and measure their cost.</li>
<li>Write the Methods paragraph that the search you ran actually supports.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read linearly and run the chunks; the searches take under a minute each, and every one is seeded. This is the <strong>R edition</strong>: the <a href="../../Python/U14_Ensembles_Tuning/index.html">Python version</a> runs the same experiments with scikit-learn, LightGBM and Optuna. Three substitutions are forced by this environment and declared where they occur. (i) The feature matrix is the <strong>32-column constructed panel of Unit 13</strong> — eleven frozen RDKit descriptors plus ratios, transforms and interactions — because R has no RDKit to compute the Python edition's 45-descriptor panel. (ii) LightGBM and CatBoost are not installable here, so the "three implementations" comparison becomes <code>ranger</code><sup class="c"><a href="#r26">[26]</a></sup>, <code>randomForest</code> and <code>xgboost</code><sup class="c"><a href="#r7">[7]</a></sup> in its two growth policies — depth-wise, which is XGBoost's own, and leaf-wise (<code>grow_policy = "lossguide"</code>), which is the idea LightGBM is built on. (iii) Optuna has no R equivalent here, so the Bayesian-style search of Section 5 is written by hand: a random-forest surrogate over the evaluated configurations plus median pruning, which is the SMAC recipe rather than Optuna's tree-structured Parzen estimator. In a fresh Colab R runtime run once: <code>install.packages(c("data.table","caret","ranger","randomForest","xgboost","rpart","patchwork","R.utils"))</code>. The unit assumes <a href="../U11_Supervised_Workflow/index.html">Unit 11</a> (preprocessing chains, baselines, leakage), <a href="../U12_Validation/index.html">Unit 12</a> (splits, nested cross-validation, model comparison) and <a href="../U13_Features_Dimensionality/index.html">Unit 13</a> (the panel used here). Decision trees themselves — how a split is chosen, what impurity means — are taught in the artificial-intelligence course; here they are a building block.</p></div>



In [ ]:
## Setup: libraries, global seed, plotting defaults, data loader.
library(data.table); library(Matrix);       library(ggplot2); library(patchwork)
library(caret);      library(ranger);       library(xgboost); library(rpart)
library(randomForest)
set.seed(0)
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10),
                panel.grid.minor = element_blank()))
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
data_path <- function(name) data_file(name)
load_data  <- function(name, ...) data.table::fread(data_path(name), ...)
cat("R", as.character(getRversion()), "| xgboost", as.character(packageVersion("xgboost")),
    "| ranger", as.character(packageVersion("ranger")), "| randomForest", as.character(packageVersion("randomForest")),
    "| rpart", as.character(packageVersion("rpart")), "| caret", as.character(packageVersion("caret")), "\n")


In [ ]:
## The modelling table: the 32-column panel of Unit 13, the target, scaffolds for the split.
df  <- load_data("rock1_chembl_curated.csv")
reg <- df[censored == FALSE]
y        <- reg$pActivity
y_cls    <- as.integer(y >= 7.0)
scaffold <- reg$scaffold_smiles
BASE <- c("MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotB",
          "AromaticRings", "RingCount", "HeavyAtoms", "FractionCSP3", "QED")
d <- as.data.table(reg[, ..BASE])
ENG <- data.table(
  polar_density  = d$TPSA / d$MolWt,          logp_per_heavy = d$MolLogP / d$HeavyAtoms,
  hbd_hba_ratio  = (d$HBD + 1) / (d$HBA + 1), aromatic_frac  = d$AromaticRings / pmax(d$RingCount, 1),
  flexibility    = d$RotB / pmax(d$HeavyAtoms, 1), log_MolWt = log(d$MolWt))
EXT <- data.table(
  sqrt_TPSA = sqrt(d$TPSA), log1p_RotB = log1p(d$RotB), logp_sq = d$MolLogP^2,
  MolWt_per_ring = d$MolWt / pmax(d$RingCount, 1), HBA_per_heavy = d$HBA / d$HeavyAtoms,
  HBD_per_heavy = d$HBD / d$HeavyAtoms, ring_density = d$RingCount / d$HeavyAtoms,
  logp_x_tpsa = d$MolLogP * d$TPSA, molwt_x_csp3 = d$MolWt * d$FractionCSP3,
  qed_x_logp = d$QED * d$MolLogP, arom_x_rotb = d$AromaticRings * d$RotB,
  hbd_x_hba = d$HBD * d$HBA, csp3_x_rings = d$FractionCSP3 * d$RingCount,
  tpsa_per_heavy = d$TPSA / d$HeavyAtoms, heavy_sq = d$HeavyAtoms^2)
P <- as.matrix(cbind(d, ENG, EXT))
fpl <- load_data("rock1_morgan_sparse.csv.gz"); ikey <- reg$inchikey; fpl <- fpl[id %in% ikey]
FP  <- sparseMatrix(i = match(fpl$id, ikey), j = fpl$bit + 1L, x = 1, dims = c(length(ikey), 2048))
rmse <- function(o, p) sqrt(mean((o - p)^2))
r2   <- function(o, p) 1 - sum((o - p)^2) / sum((o - mean(o))^2)
set.seed(0); gk  <- caret::groupKFold(scaffold, k = 5)     ## training-index sets, groups kept intact
set.seed(0); gk3 <- caret::groupKFold(scaffold, k = 3)
folds3 <- lapply(gk3, function(tr) list(tr = tr, te = setdiff(seq_along(y), tr)))
cat("panel:", dim(P), "| target sd", round(sd(y), 3), "| actives (pActivity >= 7):", sum(y_cls), "\n")
cat("all comparisons below use scaffold-grouped folds (", uniqueN(scaffold),
    "groups ) - the pessimistic protocol of Unit 12\n")
cat("5-fold test sizes:", sapply(gk, function(tr) length(y) - length(tr)),
    "| 3-fold test sizes:", sapply(folds3, function(s) length(s$te)), "\n")



<div class="box bridge"><h4>Domain context — the target, the series and the split used throughout</h4>
<p>Every number in this unit is a prediction of <strong>pActivity</strong> = \(-\log_{10}\) of the concentration at which a compound inhibits <strong>ROCK1</strong>, a human kinase the laboratory works on; larger means more potent and one unit is a tenfold change in potency. The data are a frozen snapshot of the public bioactivity database ChEMBL<sup class="c"><a href="#r29">[29]</a></sup>, curated in Unit 6. Medicinal chemists work in <strong>series</strong> — one core structure decorated many ways — so the dataset contains large families of near-twins; the computable proxy for the core is the <strong>Bemis–Murcko scaffold</strong><sup class="c"><a href="#r28">[28]</a></sup>, the molecule with its side chains stripped off. Because the question a tuned model will be asked is "is this compound from a new series worth making?", every cross-validation below groups by scaffold, the pessimistic protocol of Unit 12; the equally defensible alternative, a time split by deposition order<sup class="c"><a href="#r27">[27]</a></sup>, gives similar conclusions on this dataset. Note that <code>caret::groupKFold</code><sup class="c"><a href="#r8">[8]</a></sup> produces folds of unequal size — the printed test sizes range from roughly 400 to 1,000 compounds — because whole scaffold families must stay together.</p></div>

## 1. Bagging: averaging away variance

A deep decision tree has almost no bias and enormous variance: change a few training rows and its splits, and therefore its predictions, change completely. **Bagging** (bootstrap aggregating) exploits it<sup class="c"><a href="#r1">[1]</a></sup>: fit the same high-variance learner to many bootstrap resamples of the training set and average the predictions. Averaging \(B\) predictions with pairwise correlation \(\rho\) reduces the variance by a factor

\[ \rho + \frac{1 - \rho}{B}, \]

so the benefit is limited not by the number of trees but by how *correlated* the trees are. A **random forest**<sup class="c"><a href="#r2">[2]</a></sup> attacks precisely that term by making each split choose among a random subset of features (`mtry`), decorrelating the trees at the price of a little extra bias; extremely randomised trees push the same idea further by randomising the split thresholds too<sup class="c"><a href="#r3">[3]</a></sup>. Bagging and boosting are the two classical ways of combining weak learners<sup class="c"><a href="#r23">[23]</a></sup>; a third, *stacking*, learns a second-level model over the predictions of several first-level ones<sup class="c"><a href="#r24">[24]</a></sup> and is worth knowing but rarely worth the complexity on tables of this size. In chemistry the random forest has been the reference QSAR model since Svetnik's 2003 paper<sup class="c"><a href="#r4">[4]</a></sup>.

In R, plain bagging is a random forest with `mtry` set to all columns: `ranger(mtry = ncol(X))` fits unpruned trees on bootstrap samples and averages them, which is the definition.



In [ ]:
## A single tree, a bag of trees, and a random forest - under the scaffold split. (~10 s)
cv_scaf <- function(fitpred, folds = gk) {
  v <- sapply(folds, function(tr) {
    te <- setdiff(seq_along(y), tr)
    rmse(y[te], fitpred(P[tr, , drop = FALSE], y[tr], P[te, , drop = FALSE]))
  })
  c(mean = mean(v), sd = sd(v))
}
f_tree6 <- function(Xtr, yy, Xte) {
  m <- rpart(yy ~ ., data = cbind(as.data.frame(Xtr), yy = yy), method = "anova",
             control = rpart.control(maxdepth = 6, cp = 0, minsplit = 20, xval = 0))
  predict(m, as.data.frame(Xte))
}
f_treefull <- function(Xtr, yy, Xte) {
  m <- rpart(yy ~ ., data = cbind(as.data.frame(Xtr), yy = yy), method = "anova",
             control = rpart.control(maxdepth = 30, cp = 0, minsplit = 2, minbucket = 1, xval = 0))
  predict(m, as.data.frame(Xte))
}
f_bag <- function(Xtr, yy, Xte) {                     ## mtry = p: bagging, not a forest
  m <- ranger(x = Xtr, y = yy, num.trees = 100, mtry = ncol(Xtr), min.node.size = 1,
              seed = 1, num.threads = 4)
  predict(m, Xte)$predictions
}
f_rf <- function(Xtr, yy, Xte) {
  m <- ranger(x = Xtr, y = yy, num.trees = 300, seed = 1, num.threads = 4)
  predict(m, Xte)$predictions
}
bag_tbl <- rbindlist(lapply(
  list(c("single tree (depth 6)", "f_tree6"), c("single tree (unpruned)", "f_treefull"),
       c("bagging, 100 unpruned trees", "f_bag"), c("random forest, 300 trees", "f_rf")),
  function(z) { v <- cv_scaf(get(z[2])); data.table(model = z[1], scaffold_RMSE = v[["mean"]], sd = v[["sd"]]) }))
knitr::kable(bag_tbl[, .(model, scaffold_RMSE = round(scaffold_RMSE, 3), sd = round(sd, 3))],
             caption = "Scaffold-split RMSE in log units; the target's own standard deviation is 1.109.")



<p class="caption"><b>Table (Section 1).</b> Scaffold-split RMSE in log units. A single unpruned tree is the worst model in this unit (RMSE 1.342, well above the target's standard deviation of 1.109, i.e. worse than predicting the mean); limiting it to depth 6 brings it to 1.097; bagging a hundred copies of the <em>unpruned</em> tree drops the error to 0.986 — a 27% reduction obtained without changing the learner at all, purely by averaging. The random forest adds feature subsampling (<code>mtry</code> = 5 of 32 by default) and improves it only to 0.978, which is itself informative: with 32 heavily correlated descriptors there is little extra decorrelation to be had, and the gap between bagging and a forest is largest when a few dominant features would otherwise be chosen by every tree. Note also the fold-to-fold standard deviations: 0.08 for the ensembles against 0.04 for the single trees, because a better model has more room to vary between the unequal scaffold folds.</p>



In [ ]:
## The same table as a figure, against the predict-the-mean baseline.
bp <- copy(bag_tbl)[, model := factor(model, levels = rev(model))]
print(ggplot(bp, aes(model, scaffold_RMSE)) +
  geom_col(fill = c(rep(unname(PAL["blue"]), 2), rep("grey60", 2)), width = 0.6) +
  geom_errorbar(aes(ymin = scaffold_RMSE - sd, ymax = scaffold_RMSE + sd), width = 0.18, linewidth = 0.3) +
  geom_hline(yintercept = sd(y), linetype = "dashed", linewidth = 0.4) +
  geom_text(aes(label = sprintf("%.3f", scaffold_RMSE)), hjust = -0.25, size = 3) +
  coord_flip(ylim = c(0, 1.5)) +
  labs(x = NULL, y = "scaffold-split RMSE (log units)",
       title = "Averaging buys more than pruning"))



<p class="caption"><b>Figure 1.</b> The four models of the table, with the dashed line at the predict-the-mean baseline of 1.109. Grey bars are the single trees, blue the ensembles. The unpruned tree is <em>above</em> the baseline — a model with no bias and no restraint is worse than no model at all on a new chemotype — while averaging a hundred copies of exactly that learner lands comfortably below it. This is the clearest demonstration in the course that ensembling is not a refinement but a change of regime.</p>

### 1.1 The out-of-bag estimate: cross-validation for free

Each bootstrap resample leaves out about \(1/e \approx 37\%\) of the rows. Predicting every row with only the trees that did not see it gives the **out-of-bag** (OOB) estimate — a validation score that costs nothing extra, and which `ranger` returns as `prediction.error` (an OOB mean squared error). Its caveat is the one Unit 12 insisted on: bootstrap resampling is a *random* resampling, so the OOB estimate answers the random-split question, not the scaffold-split question.



In [ ]:
## OOB against the two cross-validated estimates it might be confused with. (~10 s)
rf_oob <- ranger(x = P, y = y, num.trees = 300, seed = 1, num.threads = 4)
set.seed(0); kf <- split(sample(seq_along(y)), rep(1:5, length.out = length(y)))
r2_random <- mean(sapply(kf, function(te) {
  tr <- setdiff(seq_along(y), te); r2(y[te], f_rf(P[tr, ], y[tr], P[te, ])) }))
r2_scaffold <- mean(sapply(gk, function(tr) {
  te <- setdiff(seq_along(y), tr); r2(y[te], f_rf(P[tr, ], y[tr], P[te, ])) }))
cat(sprintf("out-of-bag R2          : %.3f   (free, random-resampling semantics)\n",
            1 - rf_oob$prediction.error / var(y)))
cat(sprintf("random 5-fold CV R2    : %.3f\n", r2_random))
cat(sprintf("scaffold-grouped CV R2 : %.3f   <- the number to report for prospective use\n", r2_scaffold))
cat(sprintf("\nOOB mean squared error %.4f against var(y) = %.4f\n", rf_oob$prediction.error, var(y)))
cat(sprintf("probability that a given row appears in a bootstrap sample: 1 - (1 - 1/n)^n = %.3f for n = %d\n",
            1 - (1 - 1 / length(y))^length(y), length(y)))



The three numbers line up as the theory predicts: OOB (0.366) sits close to the random 5-fold estimate (0.347) and well above the scaffold estimate (0.199). OOB is an excellent, free tool for *tuning* a forest — comparing `mtry` settings, for instance, which is the hyperparameter that matters most for a forest<sup class="c"><a href="#r22">[22]</a></sup> — and a misleading one for *reporting* performance on new chemotypes. The last line is the 63.2% that makes it work: each tree sees about five eighths of the rows, so every row is out-of-bag for roughly a third of the forest.

## 2. Boosting: fitting what is left over

Boosting builds the model additively. Start from a constant, compute the residuals, fit a small tree to the residuals, add a fraction \(\nu\) (the **learning rate**) of it to the model, and repeat<sup class="c"><a href="#r5">[5]</a></sup>:

\[ F_0(x) = \bar{y}, \qquad F_m(x) = F_{m-1}(x) + \nu\, h_m(x), \qquad
   h_m \approx \text{a tree fitted to the residuals } y - F_{m-1}(x). \]

For squared error the residual *is* the negative gradient of the loss, which is why the method is called gradient boosting and why it generalises to any differentiable loss. Three ideas make it work in practice: a small learning rate (many small steps beat few large ones), shallow trees (each step corrects a little), and stochasticity — fitting each tree on a subsample of rows and columns<sup class="c"><a href="#r6">[6]</a></sup>.

### 2.1 A worked example by hand

Six points, \(x = 1 \ldots 6\) and \(y = 2, 3, 4, 7, 8, 9\). Stumps (one split), learning rate \(\nu = 0.5\).

**Step 0.** \(F_0 = \bar{y} = 5.5\) for every point. Residuals: −3.5, −2.5, −1.5, +1.5, +2.5, +3.5.

**Step 1.** The best stump on those residuals splits at \(x < 3.5\) (it separates the negative from the positive residuals) and predicts their group means, −2.5 and +2.5. The update is \(F_1 = 5.5 + 0.5 \times (\pm 2.5)\), that is 4.25 for the first three points and 6.75 for the last three. New residuals: −2.25, −1.25, −0.25, +0.25, +1.25, +2.25.

**Step 2.** The best stump on *those* splits at the same place with means ∓1.25, giving \(F_2 = 4.25 - 0.625 = 3.625\) and \(6.75 + 0.625 = 7.375\).

| x | y | F₀ | r₁ = y − F₀ | h₁ | F₁ | r₂ = y − F₁ | h₂ | F₂ |
|---|---|---|---|---|---|---|---|---|
| 1 | 2 | 5.5 | −3.5 | −2.5 | 4.25 | −2.25 | −1.25 | 3.625 |
| 2 | 3 | 5.5 | −2.5 | −2.5 | 4.25 | −1.25 | −1.25 | 3.625 |
| 3 | 4 | 5.5 | −1.5 | −2.5 | 4.25 | −0.25 | −1.25 | 3.625 |
| 4 | 7 | 5.5 | +1.5 | +2.5 | 6.75 | +0.25 | +1.25 | 7.375 |
| 5 | 8 | 5.5 | +2.5 | +2.5 | 6.75 | +1.25 | +1.25 | 7.375 |
| 6 | 9 | 5.5 | +3.5 | +2.5 | 6.75 | +2.25 | +1.25 | 7.375 |

The training RMSE falls 2.630 → 1.493 → 1.028. Two properties are already visible: the model can only produce as many distinct values as its trees have leaves (two here, however many iterations you run with the same split), and halving the learning rate would halve each correction, requiring twice as many iterations to reach the same fit — the learning rate and the number of iterations are one hyperparameter in two coordinates.



In [ ]:
## The same two steps in code, confirmed against xgboost's own implementation.
xs <- matrix(1:6, ncol = 1, dimnames = list(NULL, "x")); ys <- c(2, 3, 4, 7, 8, 9); nu <- 0.5
F <- rep(mean(ys), 6)
cat(sprintf("F0 = %s   RMSE %.3f\n", paste(sprintf("%5.3f", F), collapse = " "), rmse(ys, F)))
for (step in 1:2) {
  stump <- rpart(r ~ x, data = data.frame(x = as.numeric(xs), r = ys - F), method = "anova",
                 control = rpart.control(maxdepth = 1, cp = 0, minsplit = 2, minbucket = 1, xval = 0))
  F <- F + nu * predict(stump, data.frame(x = as.numeric(xs)))
  cat(sprintf("F%d = %s   RMSE %.3f\n", step, paste(sprintf("%5.3f", F), collapse = " "), rmse(ys, F)))
}
dm <- xgb.DMatrix(xs, label = ys, nthread = 1)
mb <- xgb.train(params = list(objective = "reg:squarederror", eta = nu, max_depth = 1, lambda = 0,
                              min_child_weight = 0, base_score = mean(ys), nthread = 1),
                data = dm, nrounds = 2, verbose = 0)
cat("\nxgboost, 2 stumps, eta 0.5, no L2 penalty:", round(predict(mb, dm), 3), "\n")



The hand calculation, the `rpart` loop and `xgboost` agree exactly: 3.625 for the three low points and 7.375 for the three high ones, with the training RMSE falling 2.630 → 1.493 → 1.028 as the table says. Nothing in a gradient-boosting library is magic; the engineering (histogram binning, leaf-wise growth, sparsity handling) only makes the same recursion fast. Two xgboost arguments had to be set to make the comparison exact — `lambda = 0` removes the L2 penalty on leaf values that xgboost applies by default, and `base_score = mean(ys)` replaces its default starting constant of 0.5 — which is a useful reminder that a library's defaults are a modelling choice, not the algorithm.

### 2.2 Watching the residuals disappear



In [ ]:
## Boosting on the real data, one stage at a time: training and scaffold-validation error per iteration.
s1  <- folds3[[1]]
dtr <- xgb.DMatrix(P[s1$tr, ], label = y[s1$tr], nthread = 4)
dte <- xgb.DMatrix(P[s1$te, ], label = y[s1$te], nthread = 4)
m400 <- xgb.train(params = list(objective = "reg:squarederror", tree_method = "hist",
                                eta = 0.05, max_depth = 6, nthread = 4),
                  data = dtr, nrounds = 400, verbose = 0)
ks <- seq(10, 400, by = 10)
stage <- rbindlist(lapply(ks, function(k) data.table(
  k = k, training = rmse(y[s1$tr], predict(m400, dtr, iterationrange = c(1, k))),
  validation = rmse(y[s1$te], predict(m400, dte, iterationrange = c(1, k))))))
best_k <- stage$k[which.min(stage$validation)]
print(ggplot(melt(stage, id.vars = "k"), aes(k, value, colour = variable)) +
  geom_vline(xintercept = best_k, linetype = "dashed", colour = "grey40", linewidth = 0.4) +
  geom_line() +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")]),
                      labels = c("training RMSE", "held-out scaffold fold"), name = NULL) +
  labs(x = "boosting iterations", y = "RMSE (log units)",
       title = "Boosting fits the training set indefinitely") + theme(legend.position = "top"))
cat(sprintf("training RMSE: %.3f at 10 trees -> %.3f at 400\n", stage$training[1], stage$training[nrow(stage)]))
cat(sprintf("held-out RMSE: %.3f at 10 -> %.3f at %d (minimum) -> %.3f at 400\n",
            stage$validation[1], min(stage$validation), best_k, stage$validation[nrow(stage)]))



<p class="caption"><b>Figure 2.</b> Training and held-out error against the number of boosting iterations on one scaffold fold. The training curve decreases monotonically, from 0.974 at ten trees to 0.342 at four hundred, and would reach zero given enough trees — boosting has no built-in stopping point, which is its main practical difference from a random forest, where adding trees never hurts. The held-out curve falls from 1.026 to a minimum of 0.992 at 60 trees (dashed line) and then drifts back up to 1.031: that drift is overfitting, and the standard remedy is <strong>early stopping</strong> on a validation fold (<code>xgb.train(..., early_stopping_rounds = 50, watchlist = list(val = dte))</code>), which is itself a hyperparameter choice that must be made inside the training fold.</p>

<div class="box key"><h4>Key idea</h4>
<p>A random forest is an ensemble of <em>independent</em> high-variance models and is tuned mainly for convenience; a boosted ensemble is a <em>sequential</em> fit whose capacity grows with every iteration, so it must be regularised — by the learning rate, the tree size, the subsampling, the L1/L2 penalties and the stopping rule. That is why boosting usually wins after tuning and often loses before it.</p></div>

## 3. The implementations available in R

Modern gradient boosting bins the features into histograms so that split-finding costs \(O(\text{bins})\) rather than \(O(\text{rows})\). The three implementations the Python edition compares — scikit-learn's `HistGradientBoosting`, XGBoost<sup class="c"><a href="#r7">[7]</a></sup> and LightGBM — are not all available here; LightGBM and CatBoost<sup class="c"><a href="#r10">[10]</a></sup> cannot be installed in this environment. What R does give is the comparison that matters most pedagogically: the two *growth policies* that distinguish the libraries from each other, both inside xgboost.

- **Depth-wise growth** (`grow_policy = "depthwise"`, XGBoost's default) expands whole levels of the tree, so `max_depth` bounds the model directly.
- **Leaf-wise growth** (`grow_policy = "lossguide"`) expands the leaf with the largest loss reduction, which is the idea LightGBM is built on; it is controlled by `max_leaves` and can produce very deep, unbalanced trees.

Alongside them, the two random-forest implementations: `ranger`<sup class="c"><a href="#r26">[26]</a></sup>, written in C++ for wide data, and the original `randomForest`, the Fortran translation of Breiman's code. They differ in speed and in default behaviour far more than in attainable accuracy<sup class="c"><a href="#r9">[9]</a></sup>; boosted trees on descriptor matrices have been a standard QSAR tool since Sheridan's systematic comparison<sup class="c"><a href="#r21">[21]</a></sup>.



In [ ]:
## Four untuned ensembles under the same scaffold cross-validation, with their cost. (~50 s)
impl_list <- list(
  "ranger forest (300)" = f_rf,
  "randomForest (300)" = function(Xtr, yy, Xte) predict(randomForest(x = Xtr, y = yy, ntree = 300), Xte),
  "xgboost depthwise (300)" = function(Xtr, yy, Xte) {
    m <- xgb.train(params = list(objective = "reg:squarederror", tree_method = "hist", eta = 0.3,
                                 max_depth = 6, nthread = 4),
                   data = xgb.DMatrix(Xtr, label = yy, nthread = 4), nrounds = 300, verbose = 0)
    predict(m, xgb.DMatrix(Xte, nthread = 4)) },
  "xgboost lossguide (300)" = function(Xtr, yy, Xte) {
    m <- xgb.train(params = list(objective = "reg:squarederror", tree_method = "hist",
                                 grow_policy = "lossguide", max_leaves = 31, max_depth = 0,
                                 eta = 0.3, nthread = 4),
                   data = xgb.DMatrix(Xtr, label = yy, nthread = 4), nrounds = 300, verbose = 0)
    predict(m, xgb.DMatrix(Xte, nthread = 4)) })
impl <- rbindlist(lapply(names(impl_list), function(nm) {
  t0 <- Sys.time(); v <- cv_scaf(impl_list[[nm]])
  trn <- mean(sapply(gk, function(tr) rmse(y[tr], impl_list[[nm]](P[tr, ], y[tr], P[tr, ]))))
  data.table(model = nm, scaffold_RMSE = v[["mean"]], sd = v[["sd"]], train_RMSE = trn,
             seconds_5_folds = as.numeric(difftime(Sys.time(), t0, units = "secs")) / 2)
}))
knitr::kable(impl[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 3) else v)],
             caption = "Untuned ensembles under the scaffold split (the time column covers five fits).")



<p class="caption"><b>Table (Section 3).</b> Untuned ensembles under the scaffold split. The two forests are indistinguishable in accuracy (0.978 and 0.980) and wildly different in cost: <code>ranger</code> fits five folds in about one second, <code>randomForest</code> in about seventeen — a fourteen-fold difference on the same algorithm, which is why every forest in this course is a <code>ranger</code>. Both boosters, run at xgboost's default learning rate of 0.3 for 300 rounds, are clearly <em>worse</em> than the forests (1.044 depth-wise, 1.052 leaf-wise) while fitting their training folds far better (train RMSE 0.300 against 0.451): the textbook signature of overfitting, produced here by accepting a default. The two growth policies differ by less than their fold-to-fold noise. The practical reading is not "forests beat boosters" — it is <strong>that untuned boosting is not a result</strong>. Section 5 tunes it.</p>

## 4. Diagnosing with curves

### 4.1 Validation curves: one hyperparameter at a time



In [ ]:
## Validation curves for the two hyperparameters that matter most. (~30 s)
xgb_fit <- function(Xtr, yy, pars) {
  nr <- pars$nrounds; pars$nrounds <- NULL
  xgb.train(params = c(list(objective = "reg:squarederror", tree_method = "hist", nthread = 4), pars),
            data = xgb.DMatrix(Xtr, label = yy, nthread = 4), nrounds = nr, verbose = 0)
}
curve_over <- function(values, make_pars, label) rbindlist(lapply(values, function(v) {
  pars <- make_pars(v)
  m <- sapply(folds3, function(s) {
    fit <- xgb_fit(P[s$tr, ], y[s$tr], pars)
    c(rmse(y[s$tr], predict(fit, xgb.DMatrix(P[s$tr, ], nthread = 4))),
      rmse(y[s$te], predict(fit, xgb.DMatrix(P[s$te, ], nthread = 4))))
  })
  data.table(parameter = label, value = v, training = mean(m[1, ]), validation = mean(m[2, ]))
}))
vc_eta <- curve_over(c(0.01, 0.03, 0.05, 0.1, 0.2, 0.4),
                     function(v) list(nrounds = 300, eta = v, max_depth = 6), "learning rate")
vc_lv  <- curve_over(c(3, 7, 15, 31, 63, 127),
                     function(v) list(nrounds = 300, eta = 0.05, grow_policy = "lossguide",
                                      max_leaves = v, max_depth = 0), "max_leaves")
p1 <- ggplot(melt(vc_eta, id.vars = c("parameter", "value"), variable.name = "set", value.name = "RMSE"),
             aes(value, RMSE, colour = set)) +
  geom_line() + geom_point(size = 1.3) + scale_x_log10() +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")]),
                      labels = c("training", "scaffold validation"), name = NULL) +
  labs(x = "learning rate (300 rounds, depth 6)", y = "RMSE (log units)", title = "Learning rate") +
  theme(legend.position = "top")
p2 <- ggplot(melt(vc_lv, id.vars = c("parameter", "value"), variable.name = "set", value.name = "RMSE"),
             aes(value, RMSE, colour = set)) +
  geom_line() + geom_point(size = 1.3) + scale_x_log10() +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")]),
                      labels = c("training", "scaffold validation"), name = NULL) +
  labs(x = "max_leaves (300 rounds, learning rate 0.05)", y = NULL, title = "Tree size") +
  theme(legend.position = "top")
print(p1 | p2)
cat(sprintf("learning rate: best %.2f at validation RMSE %.3f (training %.3f); worst setting %.3f\n",
            vc_eta$value[which.min(vc_eta$validation)], min(vc_eta$validation),
            vc_eta$training[which.min(vc_eta$validation)], max(vc_eta$validation)))
cat(sprintf("max_leaves   : best %d at validation RMSE %.3f (training %.3f); worst setting %.3f\n",
            vc_lv$value[which.min(vc_lv$validation)], min(vc_lv$validation),
            vc_lv$training[which.min(vc_lv$validation)], max(vc_lv$validation)))
cat(sprintf("gap between the best and the second-best learning rate: %.4f log units\n",
            diff(sort(vc_eta$validation))[1]))



<p class="caption"><b>Figure 3.</b> Validation curves for the learning rate (left) and tree size (right) of xgboost with 300 rounds, under three scaffold folds. The training curves fall monotonically with capacity, as they must. The validation curves tell two different stories. Tree size has the classic U shape with a minimum at 7 leaves (0.998), rising to 1.027 at 127 leaves — too few leaves underfit, too many overfit. The learning rate, over this range, is monotone: the smallest value tested (0.01) is the best (0.995) and 0.4 the worst (1.066), so the minimum lies at or below the left edge, and finding it would mean testing smaller rates with proportionally more rounds. Note how <em>flat</em> the bottom of each curve is — the difference between the best and the second-best learning rate is 0.0060 log units, far below the fold-to-fold standard deviation of about 0.08 from Section 1. This flatness is the most important practical fact about hyperparameter tuning on datasets of this size, and the reason the elaborate searches of Section 5 buy so little.</p>

### 4.2 Learning curves: would more compounds help?



In [ ]:
## Learning curves of the booster and the forest under scaffold folds. (~30 s)
lc <- rbindlist(lapply(c(0.2, 0.4, 0.6, 0.8, 1.0), function(frac) {
  nmin <- min(sapply(folds3, function(s) length(s$tr)))
  n <- floor(frac * nmin)
  v <- sapply(seq_along(folds3), function(i) {
    s <- folds3[[i]]; set.seed(100 + i); sub <- sample(s$tr, n)
    mb <- xgb_fit(P[sub, ], y[sub], list(nrounds = 300, eta = 0.05, max_depth = 6))
    mr <- ranger(x = P[sub, ], y = y[sub], num.trees = 200, seed = 1, num.threads = 4)
    c(rmse(y[sub], predict(mb, xgb.DMatrix(P[sub, ], nthread = 4))),
      rmse(y[s$te], predict(mb, xgb.DMatrix(P[s$te, ], nthread = 4))),
      rmse(y[sub], predict(mr, P[sub, ])$predictions),
      rmse(y[s$te], predict(mr, P[s$te, ])$predictions))
  })
  data.table(n = n, `xgboost training` = mean(v[1, ]), `xgboost validation` = mean(v[2, ]),
             `forest training` = mean(v[3, ]), `forest validation` = mean(v[4, ]))
}))
lcl <- melt(lc, id.vars = "n", variable.name = "curve", value.name = "RMSE")
lcl[, `:=`(model = ifelse(grepl("xgboost", curve), "xgboost", "forest"),
           kind = ifelse(grepl("training", curve), "training", "validation"))]
print(ggplot(lcl, aes(n, RMSE, colour = model, linetype = kind)) +
  geom_line() + geom_point(size = 1.2) +
  geom_hline(yintercept = sd(y), linetype = "dotted", linewidth = 0.4) +
  scale_colour_manual(values = unname(PAL[c("green", "blue")]), name = NULL) +
  scale_linetype_manual(values = c(training = "dashed", validation = "solid"), name = NULL) +
  labs(x = "training compounds", y = "RMSE (log units)",
       title = "Learning curves under the scaffold split") + theme(legend.position = "top"))
print(lc[, lapply(.SD, round, 3)])



<p class="caption"><b>Figure 4.</b> Learning curves for the booster and the forest under scaffold folds, with the predict-the-mean baseline dotted at 1.109. Both validation curves fall with training-set size and are still falling at the right-hand edge — the booster from 1.098 at 343 compounds to 1.013 at 1,718, the forest from 1.036 to 0.985 — so more compounds would still help, but note how much shallower the decline is than under the random split of Unit 11: adding compounds from the <em>same</em> chemotypes improves interpolation quickly and extrapolation slowly. The dashed training curves show the difference in character between the two models: the booster's training error <em>rises</em> with sample size (0.131 to 0.375) because 300 rounds can memorise 343 compounds and cannot memorise 1,718, while the forest's sits flat around 0.44. The gap between the dashed and solid curves of the same colour — about 0.64 log units for the booster and 0.54 for the forest at the largest size — is the variance component: these are high-capacity models on roughly 1,700 training compounds, and the scaffold split gives them no analogues to lean on.</p>

### 4.3 Bias and variance, measured

The decomposition \(\mathbb{E}[(y - \hat{f}(x))^2] = \text{bias}^2 + \text{variance} + \sigma^2\) is usually drawn, rarely computed. On simulated data, where the true function is known, it can be measured directly: draw many training sets, fit the same learner to each, and compare the *average* prediction to the truth (bias) and the *spread* of predictions around that average (variance).



In [ ]:
## Bias-variance decomposition by simulation: tree depth, and what bagging does to it. (~20 s)
truth <- function(x) sin(2 * pi * x) + 0.5 * x
gx <- seq(0, 1, length.out = 200); noise_sd <- 0.3
set.seed(0)
bv <- rbindlist(lapply(c(1, 2, 3, 5, 8, 30), function(depth) {
  Pm <- t(sapply(1:60, function(b) {
    xb <- runif(80); yb <- truth(xb) + rnorm(80, 0, noise_sd)
    m <- rpart(yb ~ xb, data = data.frame(xb = xb, yb = yb), method = "anova",
               control = rpart.control(maxdepth = depth, cp = 0,
                                       minsplit = if (depth >= 30) 2 else 20,
                                       minbucket = if (depth >= 30) 1 else 7, xval = 0))
    predict(m, data.frame(xb = gx))
  }))
  data.table(model = paste("tree depth", if (depth >= 30) "unpruned" else depth),
             bias2 = mean((colMeans(Pm) - truth(gx))^2), variance = mean(apply(Pm, 2, var)))
}))
Pf <- t(sapply(1:40, function(b) {
  xb <- runif(80); yb <- truth(xb) + rnorm(80, 0, noise_sd)
  m <- ranger(x = matrix(xb, ncol = 1, dimnames = list(NULL, "x")), y = yb, num.trees = 100,
              seed = 1, num.threads = 4, min.node.size = 1)
  predict(m, matrix(gx, ncol = 1, dimnames = list(NULL, "x")))$predictions
}))
bv <- rbind(bv, data.table(model = "forest of deep trees",
                           bias2 = mean((colMeans(Pf) - truth(gx))^2), variance = mean(apply(Pf, 2, var))))
bv[, total_error := bias2 + variance + noise_sd^2]
bvl <- melt(bv[, .(model, `bias^2` = bias2, variance, `irreducible noise` = noise_sd^2)], id.vars = "model")
print(ggplot(bvl, aes(factor(model, levels = bv$model), value, fill = variable)) +
  geom_col(width = 0.65) +
  scale_fill_manual(values = c("bias^2" = unname(PAL["orange"]), variance = unname(PAL["blue"]),
                               "irreducible noise" = "grey75"), name = NULL) +
  labs(x = NULL, y = "expected squared error at a test point",
       title = "Bias-variance decomposition, measured") +
  theme(axis.text.x = element_text(angle = 20, hjust = 1, size = 7.5), legend.position = "top"))
knitr::kable(bv[, lapply(.SD, function(v) if (is.numeric(v)) round(v, 4) else v)],
             caption = "60 simulated training sets of 80 points each; noise variance 0.09.")



<p class="caption"><b>Figure 5 and table.</b> The decomposition computed, not asserted, on 60 simulated training sets of 80 points each. A stump has bias² 0.0900 and variance 0.0219 — it cannot represent the function; an unpruned tree has bias² 0.0021 and variance 0.0971 — it represents everything, including the noise. Total error is minimised in between, and the minimum is remarkably flat: depths 3, 5 and 8 score 0.1327, 0.1321 and 0.1314 against 0.2020 for the stump and 0.1893 for the unpruned tree. The last bar is the point of Section 1: a forest of <em>deep</em> trees keeps the low bias of the unpruned tree (0.0017) and more than halves its variance (0.0457), which is what "bagging reduces variance" means quantitatively. Boosting works from the other end — it starts with high-bias stumps and reduces bias step by step, which is why its characteristic failure is overfitting late rather than underfitting early.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Repeat the simulation with 320 training points instead of 80 and with noise standard deviation 0.6 instead of 0.3. Which column changes with the sample size and which with the noise? Predict the answer before running it, then check whether the optimal depth moves.</p></div>

## 5. Hyperparameter search under a fixed budget

Three strategies, one budget of **20 configurations**, one search space, the same three scaffold folds, the same seed.

- **Grid search** evaluates a Cartesian product. With 20 evaluations you can afford two values of one parameter and three of two others — and you have decided in advance that the other four parameters keep their defaults.
- **Random search** samples configurations from distributions over *all* parameters. Bergstra and Bengio's argument is geometric: when only a few parameters matter, a grid wastes its budget re-testing the same values of the important one, while random sampling gives 20 distinct values of every parameter<sup class="c"><a href="#r11">[11]</a></sup>.
- **Bayesian optimisation** fits a surrogate model of the score as a function of the parameters and samples where it expects an improvement<sup class="c"><a href="#r12">[12]</a></sup>. Optuna implements this with a tree-structured Parzen estimator and adds **pruning**: a trial whose intermediate results are already worse than the running median is abandoned before it finishes<sup class="c"><a href="#r13">[13]</a></sup>. Hyperband-style schemes make the same bet with a different schedule<sup class="c"><a href="#r14">[14]</a></sup>. Neither Optuna nor an equivalent is installable here, so the search below is written by hand with a random-forest surrogate — the SMAC recipe — and median pruning across folds; it is twenty lines, and the twenty lines are the algorithm.



In [ ]:
## One evaluation function for all three searches: 3-fold scaffold CV of xgboost, with optional pruning.
evaluate <- function(pars, prune_at = NULL) {
  errs <- numeric(0)
  for (i in seq_along(folds3)) {
    s <- folds3[[i]]
    m <- xgb_fit(P[s$tr, ], y[s$tr], pars)
    errs <- c(errs, rmse(y[s$te], predict(m, xgb.DMatrix(P[s$te, ], nthread = 4))))
    if (!is.null(prune_at) && i < length(folds3) && mean(errs) > prune_at)
      return(c(value = NA_real_, folds = i))        ## pruned: already worse than the running median
  }
  c(value = mean(errs), folds = length(folds3))
}
sample_cfg <- function() list(
  nrounds = sample(c(100, 200, 300, 400, 600), 1), eta = sample(c(0.01, 0.02, 0.05, 0.1, 0.2), 1),
  max_depth = sample(c(2, 3, 4, 6, 8), 1), min_child_weight = sample(c(1, 5, 10, 20, 40), 1),
  subsample = sample(c(0.6, 0.8, 1.0), 1), colsample_bytree = sample(c(0.4, 0.6, 0.8, 1.0), 1),
  lambda = sample(c(0, 0.1, 1, 10), 1))
default_rmse <- evaluate(list(nrounds = 100, eta = 0.3, max_depth = 6))[["value"]]
cat(sprintf("xgboost defaults (100 rounds, eta 0.3, depth 6), 3-fold scaffold CV: RMSE %.4f\n", default_rmse))
cat("budget for every search below: 20 configurations x 3 folds\n")


In [ ]:
## Grid search: 18 configurations buys three parameters at two or three values. (~20 s)
t0 <- Sys.time()
grid <- CJ(nrounds = c(200, 600), eta = c(0.02, 0.05, 0.1), max_depth = c(3, 4, 6))
grid[, rmse := sapply(seq_len(.N), function(i)
  evaluate(list(nrounds = nrounds[i], eta = eta[i], max_depth = max_depth[i]))[["value"]])]
grid_time <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
cat(sprintf("grid search: %d configurations in %.0f s\n", nrow(grid), grid_time))
cat(sprintf("  best RMSE %.4f with nrounds = %d, eta = %.2f, max_depth = %d\n",
            min(grid$rmse), grid$nrounds[which.min(grid$rmse)], grid$eta[which.min(grid$rmse)],
            grid$max_depth[which.min(grid$rmse)]))
cat(sprintf("  spread of the %d configurations: %.4f (best) to %.4f (worst)\n",
            nrow(grid), min(grid$rmse), max(grid$rmse)))


In [ ]:
## Random search over seven parameters, same budget. (~10 s)
t0 <- Sys.time(); set.seed(0)
rand_cfgs <- lapply(1:20, function(i) sample_cfg())
rand_v <- sapply(rand_cfgs, function(cf) evaluate(cf)[["value"]])
rand_time <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
cat(sprintf("random search: 20 configurations in %.0f s | best RMSE %.4f\n", rand_time, min(rand_v)))
cat("  chosen:", paste(names(rand_cfgs[[which.min(rand_v)]]),
                       unlist(rand_cfgs[[which.min(rand_v)]]), sep = " = ", collapse = ", "), "\n")


In [ ]:
## Bayesian-style search: a random-forest surrogate over evaluated configurations + median pruning. (~20 s)
t0 <- Sys.time(); set.seed(0)
as_matrix_cfg <- function(cfgs) as.matrix(rbindlist(lapply(cfgs, as.data.table)))
hist_cfg <- list(); hist_val <- numeric(0); pruned <- 0L
for (trial in 1:20) {
  if (trial <= 5) {
    cand <- sample_cfg()                                   ## start-up trials: pure exploration
  } else {
    pool <- lapply(1:200, function(i) sample_cfg())        ## candidate pool scored by the surrogate
    ok   <- !is.na(hist_val)
    sur  <- ranger(x = as_matrix_cfg(hist_cfg[ok]), y = hist_val[ok], num.trees = 200,
                   seed = 1, num.threads = 4)
    cand <- pool[[which.min(predict(sur, as_matrix_cfg(pool))$predictions)]]
  }
  med <- if (sum(!is.na(hist_val)) >= 5) median(hist_val, na.rm = TRUE) else NULL
  res <- evaluate(cand, prune_at = med)
  hist_cfg[[trial]] <- cand; hist_val <- c(hist_val, res[["value"]])
  if (is.na(res[["value"]])) pruned <- pruned + 1L
}
bo_time <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
best_cfg <- hist_cfg[[which.min(hist_val)]]
cat(sprintf("surrogate search: 20 trials in %.0f s, %d pruned after one fold | best RMSE %.4f\n",
            bo_time, pruned, min(hist_val, na.rm = TRUE)))
cat("  chosen:", paste(names(best_cfg), unlist(best_cfg), sep = " = ", collapse = ", "), "\n")


In [ ]:
## The three searches side by side: what each bought for the same number of evaluations.
running <- cummin(ifelse(is.na(hist_val), Inf, hist_val))
summary_tbl <- data.table(search = c("defaults", "grid (18)", "random (20)", "surrogate (20)"),
                          best_RMSE = c(default_rmse, min(grid$rmse), min(rand_v), min(hist_val, na.rm = TRUE)),
                          seconds = c(0, grid_time, rand_time, bo_time))
p1 <- ggplot(data.frame(trial = seq_along(running), best = running), aes(trial, best)) +
  geom_hline(yintercept = default_rmse, linetype = "dotted", linewidth = 0.5) +
  geom_hline(yintercept = min(grid$rmse), colour = PAL[["orange"]], linetype = "dashed", linewidth = 0.4) +
  geom_hline(yintercept = min(rand_v), colour = PAL[["green"]], linetype = "dashed", linewidth = 0.4) +
  geom_step(colour = PAL[["blue"]]) +
  labs(x = "completed trial", y = "best 3-fold scaffold RMSE so far",
       title = "Surrogate search history (dashed: grid, random; dotted: defaults)")
p2 <- ggplot(summary_tbl, aes(factor(search, levels = search), best_RMSE)) +
  geom_col(fill = c("grey60", unname(PAL[c("orange", "green", "blue")])), width = 0.6) +
  geom_text(aes(label = sprintf("%.4f", best_RMSE)), vjust = -0.4, size = 3) +
  coord_cartesian(ylim = c(min(summary_tbl$best_RMSE) - 0.02, max(summary_tbl$best_RMSE) + 0.01)) +
  labs(x = NULL, y = "best 3-fold scaffold RMSE", title = "Same budget, three strategies")
print(p1 | p2)
knitr::kable(summary_tbl[, .(search, best_RMSE = round(best_RMSE, 4), seconds = round(seconds))],
             caption = "What each search bought for 18-20 evaluations.")



<p class="caption"><b>Figure 6 and table.</b> One budget, three strategies, on the same folds. The grid improves on the defaults (1.0355 → 0.9945) but only by exploring three parameters and leaving the other four alone, and it needed the longest wall-clock time because its 600-round configurations are expensive. Random search, sampling all seven parameters, does better with the same number of fits and in half the time (0.9892) — the Bergstra–Bengio result reproduced on our data<sup class="c"><a href="#r11">[11]</a></sup>. The surrogate search does better still (0.9834), and because the median pruner abandoned 2 hopeless trials after a single fold it spent part of its budget on new configurations rather than on finishing bad ones. <strong>And the whole exercise moves the RMSE by about 0.05 log units</strong> — comparable to the fold-to-fold standard deviation of 0.06–0.08 and far smaller than the 0.15 log units that Unit 11 bought by changing the representation. That is the honest summary of hyperparameter tuning for a dataset of this size: it is worth doing once, properly and reproducibly, and it is not where the accuracy comes from.</p>

### 5.1 The number you may report

The best value of a search is the best of 20 noisy estimates: the winner's curse of Unit 12 applies, and quoting it as the model's performance is the most common tuning error in the literature<sup class="c"><a href="#r15">[15]</a></sup>. The honest estimate puts the *whole search* inside an outer loop — nested cross-validation with a fixed inner budget, so that the thing being assessed is "xgboost plus an 8-configuration random search", not "the configuration that happened to win".



In [ ]:
## Nested cross-validation: the search is re-run inside every outer fold. (~15 s)
nested <- t(sapply(seq_along(folds3), function(i) {
  s <- folds3[[i]]
  set.seed(200 + i)
  inner <- caret::groupKFold(scaffold[s$tr], k = 2)
  cfgs  <- lapply(1:8, function(j) sample_cfg())
  sc <- sapply(cfgs, function(cf) mean(sapply(inner, function(itr) {
    tr2 <- s$tr[itr]; te2 <- s$tr[setdiff(seq_along(s$tr), itr)]
    m <- xgb_fit(P[tr2, ], y[tr2], cf)
    rmse(y[te2], predict(m, xgb.DMatrix(P[te2, ], nthread = 4)))
  })))
  cf <- cfgs[[which.min(sc)]]
  m  <- xgb_fit(P[s$tr, ], y[s$tr], cf)
  c(outer = rmse(y[s$te], predict(m, xgb.DMatrix(P[s$te, ], nthread = 4))),
    inner = min(sc), eta = cf$eta, max_depth = cf$max_depth)
}))
cat(sprintf("single-level search, best reported score   : %.4f   <- optimistic: the minimum of 20 estimates\n",
            min(hist_val, na.rm = TRUE)))
cat(sprintf("nested CV, outer estimate of the procedure : %.4f  (folds %s)\n",
            mean(nested[, "outer"]), paste(round(nested[, "outer"], 3), collapse = ", ")))
cat(sprintf("difference (optimism of the quoted best)   : %.4f log units\n",
            mean(nested[, "outer"]) - min(hist_val, na.rm = TRUE)))
cat(sprintf("inner best scores (trained on 2/3 of the data, 2 folds): %s - pessimistic for the opposite reason\n",
            paste(round(nested[, "inner"], 3), collapse = ", ")))
cat("learning rate and depth chosen in each outer fold:", nested[, "eta"], "|", nested[, "max_depth"], "\n")



Three numbers, three meanings. The single-level search reports 0.9834 — the best of twenty noisy estimates on the same folds it optimised. The nested estimate of the whole procedure is 1.0018, so the quoted best is optimistic by 0.0184 log units, and the honest figure is barely better than the untuned defaults (1.0355). The inner scores are *worse* than both (1.000 to 1.046), because each inner search trains on half of two thirds of the data: a smaller training set gives a worse model, which is the standard pessimistic bias of cross-validation and the reason the inner score must never be reported either. The number that belongs in a paper is the nested one, together with the search space and the budget.

<div class="box warn"><h4>What tuning cannot fix</h4>
<p>Compare the whole of Section 5 with Unit 11's table: changing the representation from eleven descriptors to 2,048 fingerprint bits moved RMSE by about 0.15 log units, and changing the split from random to scaffold moved it by about 0.06. Twenty tuning trials moved it by 0.05, of which nested cross-validation says only part is real. Spend the project's effort in that order — data, representation, validation design — and tune last.</p></div>

## 6. Class imbalance: weights, thresholds, and what each costs

Make the ROCK1 classification problem realistic for screening by sub-sampling the actives to 5%, then apply the two standard remedies: weighting the classes in the loss (`scale_pos_weight`) and moving the decision threshold<sup class="c"><a href="#r18">[18]</a></sup>. A third family, synthetic over-sampling of the minority class such as SMOTE<sup class="c"><a href="#r25">[25]</a></sup>, interpolates new positives between existing ones; it is popular, it must be applied inside the training fold only, and on molecular data it manufactures feature vectors that correspond to no molecule.



In [ ]:
## Class weighting and threshold tuning on a 5%-active version of the problem. (~10 s)
cmat <- function(pred, obs) c(TP = sum(pred == 1 & obs == 1), FP = sum(pred == 1 & obs == 0),
                              FN = sum(pred == 0 & obs == 1), TN = sum(pred == 0 & obs == 0))
mcc <- function(k) {
  den <- sqrt(as.numeric(k[["TP"]] + k[["FP"]]) * (k[["TP"]] + k[["FN"]]) *
              (k[["TN"]] + k[["FP"]]) * (k[["TN"]] + k[["FN"]]))
  if (den == 0) return(0)
  (k[["TP"]] * k[["TN"]] - k[["FP"]] * k[["FN"]]) / den
}
auc_roc <- function(o, p) { r <- rank(p); n1 <- sum(o == 1); n0 <- sum(o == 0)
  (sum(r[o == 1]) - n1 * (n1 + 1) / 2) / (n1 * n0) }
auc_pr <- function(o, p) { or <- order(p, decreasing = TRUE); yt <- o[or]; tp <- cumsum(yt)
  sum(diff(c(0, tp / sum(yt))) * (tp / seq_along(yt))) }
set.seed(0)
act <- which(y_cls == 1); ina <- which(y_cls == 0)
rare <- c(ina, sample(act, round(0.05 / 0.95 * length(ina))))
set.seed(0)
te_i <- rare[caret::createDataPartition(factor(y_cls[rare]), p = 0.25, list = FALSE)]
tr_i <- setdiff(rare, te_i)
ths <- seq(0.02, 0.9, length.out = 45)
imb <- rbindlist(lapply(c(1, 19), function(spw) {
  m <- xgb.train(params = list(objective = "binary:logistic", tree_method = "hist", eta = 0.1,
                               max_depth = 4, scale_pos_weight = spw, nthread = 4),
                 data = xgb.DMatrix(P[tr_i, ], label = y_cls[tr_i], nthread = 4), nrounds = 200, verbose = 0)
  p  <- predict(m, xgb.DMatrix(P[te_i, ], nthread = 4))
  mc <- sapply(ths, function(t) mcc(cmat(as.integer(p >= t), y_cls[te_i])))
  data.table(model = if (spw == 1) "unweighted" else "scale_pos_weight = 19",
             ROC_AUC = auc_roc(y_cls[te_i], p), PR_AUC = auc_pr(y_cls[te_i], p),
             `MCC@0.5` = mcc(cmat(as.integer(p >= 0.5), y_cls[te_i])),
             best_threshold = ths[which.max(mc)], `MCC@best` = max(mc),
             Brier = mean((p - y_cls[te_i])^2), curve = list(data.table(threshold = ths, MCC = mc)))
}))
print(ggplot(rbindlist(lapply(seq_len(nrow(imb)), function(i) imb$curve[[i]][, model := imb$model[i]])),
       aes(threshold, MCC, colour = model)) +
  geom_vline(xintercept = 0.5, linetype = "dotted", colour = "grey40") +
  geom_line() +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")]), name = NULL) +
  labs(x = "decision threshold", y = "MCC",
       title = sprintf("5%% actives (test prevalence %.3f): the threshold matters more than the weights",
                       mean(y_cls[te_i]))) + theme(legend.position = "top"))
knitr::kable(imb[, .(model, ROC_AUC = round(ROC_AUC, 3), PR_AUC = round(PR_AUC, 3),
                     `MCC@0.5` = round(`MCC@0.5`, 3), best_threshold = round(best_threshold, 2),
                     `MCC@best` = round(`MCC@best`, 3), Brier = round(Brier, 4))],
             caption = "Two ways to treat a 5%-active problem; 488 test compounds, 25 of them active.")



<p class="caption"><b>Figure 7 and table.</b> Two ways to treat a 5%-active problem. The weighted and unweighted models have practically the same discrimination (ROC-AUC 0.681 and 0.683, PR-AUC 0.205 and 0.212), so class weighting does <em>not</em> help the model separate actives from inactives — it rescales the predicted probabilities, which is why the Brier score gets markedly <em>worse</em> (0.0473 to 0.0633) and why the weighted model's best operating point moves all the way to a threshold of 0.90. What changes the usable performance is the threshold itself: for the unweighted model MCC rises from 0.185 at the default 0.5 to 0.311 at its optimum. Note also how jagged both curves are — with 25 positives in the test set, moving the threshold by 0.02 can swing MCC by a tenth, which is a reminder that an "optimal threshold" chosen on a small validation set is itself a noisy estimate and should be selected inside the resampling. The lesson, which the clinical-prediction literature states forcefully<sup class="c"><a href="#r18">[18]</a></sup>, is that imbalance corrections are a threshold decision in disguise: fit an unweighted, calibrated model and then choose the operating point from the costs, rather than distorting the likelihood and keeping an arbitrary cut-off.</p>

## 7. Monotonic constraints: putting chemistry into the model

Sometimes the direction of an effect is known in advance and the data are too sparse to establish it. A model that predicts a *lower* potency for a compound more similar to the known actives is contradicting the similar-property principle, and a referee will say so. Gradient-boosting implementations can enforce monotonicity in a feature: every split is constrained so that the prediction is non-decreasing (or non-increasing) along it<sup class="c"><a href="#r19">[19]</a></sup>; in xgboost the argument is `monotone_constraints`, a parenthesised vector with one entry per column.

The experiment below adds one feature — the Tanimoto similarity to the 100 most potent *training* compounds, recomputed inside each fold so that no test information leaks — and fits the booster with and without the constraint that predictions increase with it.



In [ ]:
## A domain-motivated monotone feature, with and without the constraint. (~10 s)
tanimoto <- function(A, B) {
  C <- as.matrix(Matrix::tcrossprod(A, B))
  a <- Matrix::rowSums(A); b <- Matrix::rowSums(B)
  C / (outer(a, b, "+") - C)
}
sim_to_top <- function(train_idx, query_idx, n_top = 100) {
  top <- train_idx[order(-y[train_idx])[1:n_top]]        ## most potent TRAINING compounds only
  apply(tanimoto(FP[query_idx, , drop = FALSE], FP[top, , drop = FALSE]), 1, max)
}
CST <- paste0("(", paste(c(rep(0, ncol(P)), 1), collapse = ","), ")")
mono <- rbindlist(lapply(c("unconstrained", "monotone increasing in similarity"), function(lab) {
  v <- sapply(folds3, function(s) {
    Xtr <- cbind(P[s$tr, ], sim = sim_to_top(s$tr, s$tr))
    Xte <- cbind(P[s$te, ], sim = sim_to_top(s$tr, s$te))
    pars <- list(objective = "reg:squarederror", tree_method = "hist", eta = 0.05, max_depth = 6, nthread = 4)
    if (lab != "unconstrained") pars$monotone_constraints <- CST
    m <- xgb.train(params = pars, data = xgb.DMatrix(Xtr, label = y[s$tr], nthread = 4),
                   nrounds = 150, verbose = 0)
    rmse(y[s$te], predict(m, xgb.DMatrix(Xte, nthread = 4)))
  })
  data.table(model = lab, scaffold_RMSE = mean(v), sd = sd(v))
}))
knitr::kable(mono[, .(model, scaffold_RMSE = round(scaffold_RMSE, 3), sd = round(sd, 3))],
             caption = "The cost of a constraint the chemistry requires.")


In [ ]:
## The partial dependence of the similarity feature, with and without the constraint.
s1 <- folds3[[1]]
Xtr1 <- cbind(P[s1$tr, ], sim = sim_to_top(s1$tr, s1$tr))
grid_s <- seq(min(Xtr1[, "sim"]), max(Xtr1[, "sim"]), length.out = 12)
pdp <- rbindlist(lapply(c("unconstrained", "monotone"), function(lab) {
  pars <- list(objective = "reg:squarederror", tree_method = "hist", eta = 0.05, max_depth = 6, nthread = 4)
  if (lab == "monotone") pars$monotone_constraints <- CST
  m <- xgb.train(params = pars, data = xgb.DMatrix(Xtr1, label = y[s1$tr], nthread = 4),
                 nrounds = 150, verbose = 0)
  data.table(model = lab, sim = grid_s,
             pd = sapply(grid_s, function(g) {        ## partial dependence, by its definition
               Xg <- Xtr1; Xg[, "sim"] <- g
               mean(predict(m, xgb.DMatrix(Xg, nthread = 4)))
             }))
}))
print(ggplot(pdp, aes(sim, pd, colour = model)) + geom_line() + geom_point(size = 1.2) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "Tanimoto similarity to the 100 most potent training compounds",
       y = "partial dependence of predicted pActivity",
       title = "A constraint the chemistry requires") + theme(legend.position = "top"))
print(dcast(pdp, sim ~ model, value.var = "pd")[, lapply(.SD, round, 3)])



<p class="caption"><b>Figure 8 and tables.</b> The partial-dependence curve of the similarity feature, with and without the monotone constraint, and the cross-validated cost of imposing it. The unconstrained curve is already close to monotone — the data do support the similar-property principle, and the predicted potency climbs from 6.03 at similarity 0.13 to 8.47 at similarity 1.0 — but it dips between the third and fourth grid points (6.120 then 6.118), where few compounds live, and such dips are noise the model would happily extrapolate. The constrained model removes them and is marginally <em>better</em> rather than worse: scaffold RMSE 0.921 against 0.926 unconstrained, with a comparable fold-to-fold spread (0.013 against 0.011) — the constraint costs nothing measurable here. This is the general shape of a good constraint: it encodes knowledge the data cannot contradict, buys robustness where data are thin, and should be reported as a modelling choice in the Methods. Note too that adding this one feature improves the booster from 1.013 (Section 4.2) to 0.921 — one well-chosen feature was worth more than the entire search of Section 5.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Impose the <em>wrong</em> constraint: require the prediction to be monotone <strong>decreasing</strong> in the same similarity feature (replace the final <code>1</code> in <code>CST</code> by <code>-1</code>). Measure the scaffold RMSE and plot the partial dependence. How large is the penalty, and what does the size of that penalty tell you about how strongly the data support the correct direction?</p></div>

## 8. The final model, and what it learned



In [ ]:
## Refit the selected configuration on one scaffold-split training set and score the held-out fold.
nr <- best_cfg$nrounds; pars_final <- best_cfg; pars_final$nrounds <- NULL
final_model <- xgb.train(params = c(list(objective = "reg:squarederror", tree_method = "hist", nthread = 4),
                                    pars_final),
                         data = xgb.DMatrix(P[s1$tr, ], label = y[s1$tr], nthread = 4),
                         nrounds = nr, verbose = 0)
pred_f <- predict(final_model, xgb.DMatrix(P[s1$te, ], nthread = 4))
imp <- xgb.importance(model = final_model, feature_names = colnames(P))[1:12][order(Gain)]
p1 <- ggplot(data.frame(pred = pred_f, obs = y[s1$te]), aes(pred, obs)) +
  geom_abline(linetype = "dashed", linewidth = 0.4) +
  geom_point(size = 0.7, alpha = 0.4, colour = PAL[["blue"]]) +
  coord_equal() +
  labs(x = "predicted pActivity", y = "observed pActivity",
       title = sprintf("Held-out scaffold fold (n = %d, RMSE %.2f)", length(s1$te), rmse(y[s1$te], pred_f)))
p2 <- ggplot(imp, aes(factor(Feature, levels = Feature), Gain)) +
  geom_col(fill = PAL[["green"]], width = 0.7) + coord_flip() +
  labs(x = NULL, y = "share of total split gain", title = "What the tuned booster splits on") +
  theme(axis.text.y = element_text(size = 7.5))
print(p1 | p2)
cat(sprintf("tuned model on a held-out scaffold fold: RMSE %.3f | predict-the-mean baseline %.3f\n",
            rmse(y[s1$te], pred_f), rmse(y[s1$te], rep(mean(y[s1$tr]), length(s1$te)))))
cat("top five columns by gain:", paste(rev(imp$Feature)[1:5], round(rev(imp$Gain)[1:5], 3),
                                       sep = " = ", collapse = ", "), "\n")



<p class="caption"><b>Figure 9.</b> Left: observed against predicted potency for the held-out scaffold fold under the tuned configuration (RMSE 0.977 against a predict-the-mean baseline of 1.077); the cloud is strongly tilted towards the mean, exactly as in Unit 11, and on a fold this hard the model is a weak ranking device rather than a potency calculator. Right: the share of total split gain taken by each panel column. The ranking is dominated by <code>flexibility</code> (rotatable bonds per heavy atom, 0.113 of the total gain), <code>logp_x_tpsa</code>, <code>MolWt_per_ring</code> and <code>HBA_per_heavy</code> — shape and polarity <em>ratios</em> rather than raw size, which is a reasonable chemical story for a kinase hinge binder and a vindication of the feature construction of Unit 13. Read it with that unit's caution: gain importances are not causal, they are split frequencies weighted by improvement, and correlated columns divide the credit among themselves. Unit 16 replaces this bar chart with Shapley values<sup class="c"><a href="#r20">[20]</a></sup>, which at least attribute consistently.</p>

## 9. The paragraph that goes in the paper

A Methods section must let a reader reproduce the search, not merely admire the result; for clinical prediction models the same requirements are now codified in TRIPOD+AI<sup class="c"><a href="#r30">[30]</a></sup>. The chunk below writes it from the objects created above, so no number in it can drift from the experiment.



In [ ]:
## The Methods paragraph, generated from the actual run.
methods <- sprintf(
"Gradient-boosted regression trees (xgboost %s) were trained to predict pActivity (-log10 M) from a
%d-column descriptor panel (11 RDKit descriptors from the curated table plus 21 ratios, transforms and
interactions; Unit 13) for %d ROCK1 inhibitors with at least one exact measurement (curated ChEMBL
snapshot). Because the intended use is prospective prediction on new chemotypes, all cross-validation was
grouped by Bemis-Murcko scaffold (%d groups); random resampling was used only for the out-of-bag
comparison, which is reported separately. Hyperparameters were optimised with a random-forest surrogate
search (20 trials, seed 0, median pruning across folds after 5 start-up trials; %d trials pruned) over the
space: nrounds 100-600, eta 0.01-0.2, max_depth 2-8, min_child_weight 1-40, subsample 0.6-1.0,
colsample_bytree 0.4-1.0, lambda 0-10, with 3 scaffold folds per trial. The selected configuration was
%s, with a cross-validated RMSE of %.3f log units. Because that value is the minimum of 20 correlated
estimates, performance is reported from nested cross-validation in which the search was repeated inside
each of three outer scaffold folds (8 random configurations per inner search, 2 inner folds):
RMSE %.3f +- %.3f log units, against %.3f for untuned xgboost, %.3f for an untuned ranger random forest
and %.3f for predicting the training mean. Predictions were constrained to increase monotonically with the
Tanimoto similarity to the 100 most potent training compounds, a feature recomputed within each training
fold (scaffold RMSE %.3f constrained against %.3f unconstrained). All seeds were fixed and the complete
search history is in the accompanying notebook.",
  as.character(packageVersion("xgboost")), ncol(P), length(y), uniqueN(scaffold), pruned,
  paste(names(best_cfg), unlist(best_cfg), sep = "=", collapse = ", "), min(hist_val, na.rm = TRUE),
  mean(nested[, "outer"]), sd(nested[, "outer"]), default_rmse,
  bag_tbl$scaffold_RMSE[bag_tbl$model == "random forest, 300 trees"], sd(y),
  mono$scaffold_RMSE[2], mono$scaffold_RMSE[1])
cat(methods, "\n")



Read that paragraph against what a typical paper says — "hyperparameters were optimised by grid search and the best model achieved \(R^2\) = 0.X" — and the difference is the whole unit: the split, the budget, the space, the pruning, the seed, the nested estimate, the baselines, and the constraint.

<div class="box bridge"><h4>Bridge — where this unit sits</h4>
<p><a href="../U11_Supervised_Workflow/index.html">Unit 11 (R)</a> supplies the preprocessing discipline every search here depends on, and <a href="../U12_Validation/index.html">Unit 12 (R)</a> the scaffold protocol, the nested-CV argument and the corrected paired test you would use to compare the tuned booster against the forest; <a href="../U13_Features_Dimensionality/index.html">Unit 13 (R)</a> built the panel and showed why tree ensembles tolerate irrelevant columns. The <a href="../index.html">R-edition index</a> lists the remaining units — Unit 16 opens the tuned ensemble with Shapley values, Unit 17 attaches conformal prediction intervals to it, Unit 18 asks whether a neural network can beat it on this table (the literature says usually not<sup class="c"><a href="#r16">[16]</a></sup>), Unit 25 moves a search of this shape to a cluster, and Unit 27 assembles the whole chain. The Python counterpart of this unit, with LightGBM and Optuna, is <a href="../../Python/U14_Ensembles_Tuning/index.html">here</a>. Decision trees, boosting as an optimisation algorithm and the bias-variance theory are treated from the machine-learning side in the artificial-intelligence course, whose R edition covers trees and forests in <a href="../../../BGM411/R/U4_Trees_Random_Forest/index.html">U4</a>; the QSAR tradition these models grew out of is <a href="../../../Cheminformatics/U13_ML_Properties/index.html">Unit 13 of the cheminformatics course</a>.</p></div>

## 10. Summary

You should now be able to explain, without notes:

- how bagging reduces variance, why decorrelating trees matters, and what the out-of-bag estimate does and does not measure;
- the gradient-boosting recursion, traced by hand, and the role of the learning rate, the tree size, subsampling and regularisation;
- why untuned boosting can lose to an untuned forest, and why tuned boosting usually wins;
- how to read a validation curve and a learning curve, and what a measured bias–variance decomposition looks like;
- the three search strategies, why random beats grid at equal budget, what pruning buys, and how to report a search;
- why the best score of a search is not a performance estimate, and how nested cross-validation repairs it;
- what class weighting actually does under imbalance, and why the threshold is the real lever;
- when a monotonic constraint is justified and how to measure its cost;
- what belongs in the Methods paragraph.

The numbers to remember from this document: bagging 100 unpruned trees cuts the scaffold RMSE of a single unpruned tree from 1.342 to 0.986, below the predict-the-mean baseline of 1.109; untuned xgboost at its default learning rate is worse than either forest (1.044 against 0.978); twenty trials of surrogate search improve it from 1.0355 to 0.9834 while nested cross-validation puts the honest figure at 1.0018; class weighting leaves ROC-AUC at about 0.68 and worsens the Brier score from 0.047 to 0.063, while moving the threshold raises MCC from 0.185 to 0.311; and one monotone similarity feature was worth more than the entire hyperparameter search.

## 11. Self-assessment

<details><summary>1. Why does averaging 100 unpruned trees beat one unpruned tree by so much?</summary>Because an unpruned tree has very low bias and very high variance, and averaging B models with pairwise correlation ρ reduces the variance to ρ + (1−ρ)/B of a single model's. The simulation in Section 4.3 measures it: variance 0.0971 for one deep tree, 0.0457 for a forest of them, with bias² essentially unchanged (0.0021 against 0.0017).</details>

<details><summary>2. Your forest's OOB \(R^2\) is 0.366 and your scaffold-split \(R^2\) is 0.199. Which do you report?</summary>The scaffold number, if the claim is about new chemotypes. OOB resamples rows at random, so it estimates the random-split quantity — and indeed it sits next to the random 5-fold figure of 0.347. OOB is excellent for cheap internal comparisons such as choosing <code>mtry</code>, and misleading as a headline.</details>

<details><summary>3. You halve the learning rate of a boosted model. What else must change?</summary>The number of iterations, roughly doubled — the two are one hyperparameter in two coordinates. Everything else equal, a smaller learning rate with proportionally more trees usually generalises slightly better and costs proportionally more compute; our validation curve had not found its minimum even at eta = 0.01 with 300 rounds.</details>

<details><summary>4. Why does adding trees never hurt a random forest but can hurt a boosted model?</summary>Forest trees are fitted independently to bootstrap samples, so adding them only improves the average. Boosting fits each tree to the current residuals, so capacity grows with every iteration and late iterations start fitting noise — here the held-out error reached its minimum at 90 trees and was 0.018 log units worse by 400.</details>

<details><summary>5. Grid and random search get the same 20 evaluations. Why does random usually win?</summary>Because a grid spends its budget on a few values of every parameter jointly, re-testing the same value of the important parameter many times, while random sampling gives 20 distinct values of each parameter and covers the dimensions that matter. The advantage grows with the number of parameters that do not matter — here 0.9945 against 0.9892, in half the time.</details>

<details><summary>6. What does median pruning do, and what is the risk?</summary>It stops a trial whose running mean after one or two folds is already worse than the median of completed trials, freeing budget for new configurations. The risk is pruning a configuration that starts badly on an unlucky fold and would finish well, which is why a warm-up of a few start-up trials and at least one completed fold is standard; in our run 2 of 20 trials were pruned.</details>

<details><summary>7. A paper reports "best cross-validated RMSE from a 500-point grid search". What is wrong with that number?</summary>It is the minimum of 500 correlated noisy estimates, so it is biased optimistically — the winner's curse. Our much smaller search is optimistic by 0.0184 log units; with 500 candidates and less data the bias is far larger. The remedy is nested cross-validation or a genuinely untouched test set.</details>

<details><summary>8. Your nested CV gives a worse score than your single-level search and your inner folds give a worse score still. Is something broken?</summary>No. The single-level best is optimistic (minimum of many estimates); the nested outer estimate is the honest one (1.0018 here); the inner scores are pessimistic because they train on a fraction of a fraction of the data (1.000 to 1.046). Three estimators of three different quantities.</details>

<details><summary>9. When does class weighting help?</summary>When the loss really should weight the classes differently — for instance when the training sample under-represents the positive class relative to deployment. It does not improve ranking (ROC-AUC moved from 0.681 to 0.683) and it degrades calibration (Brier 0.047 to 0.063), so if the goal is a usable probability plus a decision, prefer an unweighted model with a chosen threshold.</details>

<details><summary>10. Give an example of a monotonic constraint you could justify in this laboratory, and one you could not.</summary>Justified: predicted potency non-decreasing in similarity to known actives, which cost nothing here (0.921 against 0.926 unconstrained). Not justified: potency monotone in molecular weight — larger is better only up to a point, and forcing monotonicity would encode a false belief that the data would otherwise contradict.</details>

<details><summary>11. You have a week of compute. Where do you spend it?</summary>Not on tuning. On more or better data, a better representation, and a validation design that answers the real question — in this block the representation moved the error by about 0.15 log units and one engineered similarity feature by about 0.09, against 0.05 for twenty tuning trials of which nested CV says only part is real. Tune once at the end, with a stated budget and a nested estimate.</details>

<details><summary>12. What minimum information must a Methods section give about a hyperparameter search?</summary>The search space with its ranges and scales, the budget (number of configurations and the resampling used per configuration), the algorithm and its seed, the pruning or early-stopping rule, the selected configuration, and an estimate of performance that did not come from the search itself — exactly the clauses generated in Section 9.</details>

## Glossary

<dl class="gloss">
<dt>Bagging</dt><dd>Fitting one learner to many bootstrap resamples and averaging, to reduce variance; in R, <code>ranger(mtry = ncol(X))</code>.</dd>
<dt>Bias–variance decomposition</dt><dd>Splitting expected squared error into systematic error, sensitivity to the training sample, and irreducible noise.</dd>
<dt>Boosting</dt><dd>Additive fitting in which each new small model is fitted to the residuals (negative gradient) of the current ensemble.</dd>
<dt>colsample / subsample</dt><dd>Fractions of features and rows used per tree; stochastic regularisation of a boosted ensemble.</dd>
<dt>Early stopping</dt><dd>Halting boosting when a validation score stops improving; a hyperparameter chosen inside the training fold.</dd>
<dt>Learning rate (shrinkage, <code>eta</code>)</dt><dd>The factor by which each boosting step is multiplied before being added to the ensemble.</dd>
<dt>Leaf-wise vs depth-wise growth</dt><dd><code>grow_policy = "lossguide"</code> expands the leaf with the largest loss reduction; <code>"depthwise"</code> grows whole levels.</dd>
<dt>Monotonic constraint</dt><dd>A restriction forcing predictions to be non-decreasing (or non-increasing) in a chosen feature.</dd>
<dt>mtry</dt><dd>The number of candidate features considered at each split of a random forest; its main hyperparameter.</dd>
<dt>Nested cross-validation</dt><dd>An outer loop assessing a procedure whose inner loop performs the hyperparameter selection.</dd>
<dt>Out-of-bag (OOB) estimate</dt><dd>Prediction of each row by the bootstrap trees that did not see it; a free random-resampling validation score.</dd>
<dt>Partial dependence</dt><dd>The average prediction as one feature is swept across its range with the others held at their observed values.</dd>
<dt>Pruning (of a search)</dt><dd>Abandoning a trial whose intermediate results are already uncompetitive.</dd>
<dt>Random forest</dt><dd>Bagged trees whose splits consider a random subset of features, reducing inter-tree correlation.</dd>
<dt><code>scale_pos_weight</code></dt><dd>A loss weight compensating class imbalance; it rescales probabilities rather than improving ranking.</dd>
<dt>Search budget</dt><dd>The number of configurations and the resampling cost per configuration that a search is allowed.</dd>
<dt>Surrogate model</dt><dd>A cheap model of the score as a function of the hyperparameters, used to choose the next configuration to try.</dd>
<dt>Winner's curse</dt><dd>Upward bias of the best of several noisy estimates, which makes a search's own best score optimistic.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Breiman L. Bagging predictors. <em>Mach Learn</em>. 1996;24:123-140. <a href="https://doi.org/10.1007/BF00058655">doi:10.1007/BF00058655</a></li>
<li id="r2">Breiman L. Random forests. <em>Mach Learn</em>. 2001;45:5-32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r3">Geurts P, Ernst D, Wehenkel L. Extremely randomized trees. <em>Mach Learn</em>. 2006;63:3-42. <a href="https://doi.org/10.1007/s10994-006-6226-1">doi:10.1007/s10994-006-6226-1</a></li>
<li id="r4">Svetnik V, Liaw A, Tong C, Culberson JC, Sheridan RP, Feuston BP. Random forest: a classification and regression tool for compound classification and QSAR modeling. <em>J Chem Inf Comput Sci</em>. 2003;43(6):1947-1958. <a href="https://doi.org/10.1021/ci034160g">doi:10.1021/ci034160g</a></li>
<li id="r5">Friedman JH. Greedy function approximation: a gradient boosting machine. <em>Ann Stat</em>. 2001;29(5):1189-1232. <a href="https://doi.org/10.1214/aos/1013203451">doi:10.1214/aos/1013203451</a></li>
<li id="r6">Friedman JH. Stochastic gradient boosting. <em>Comput Stat Data Anal</em>. 2002;38(4):367-378. <a href="https://doi.org/10.1016/S0167-9473(01)00065-2">doi:10.1016/S0167-9473(01)00065-2</a></li>
<li id="r7">Chen T, Guestrin C. XGBoost: a scalable tree boosting system. In: <em>Proc. 22nd ACM SIGKDD</em>. ACM; 2016:785-794. <a href="https://doi.org/10.1145/2939672.2939785">doi:10.1145/2939672.2939785</a></li>
<li id="r8">Kuhn M. Building predictive models in R using the caret package. <em>J Stat Softw</em>. 2008;28(5):1-26. <a href="https://doi.org/10.18637/jss.v028.i05">doi:10.18637/jss.v028.i05</a></li>
<li id="r9">Bentéjac C, Csörgő A, Martínez-Muñoz G. A comparative analysis of gradient boosting algorithms. <em>Artif Intell Rev</em>. 2021;54:1937-1967. <a href="https://doi.org/10.1007/s10462-020-09896-5">doi:10.1007/s10462-020-09896-5</a></li>
<li id="r10">Prokhorenkova L, Gusev G, Vorobev A, Dorogush AV, Gulin A. CatBoost: unbiased boosting with categorical features. <em>arXiv</em>. 2018. <a href="https://doi.org/10.48550/arXiv.1706.09516">doi:10.48550/arXiv.1706.09516</a></li>
<li id="r11">Bergstra J, Bengio Y. Random search for hyper-parameter optimization. <em>J Mach Learn Res</em>. 2012;13:281-305. <a href="https://www.jmlr.org/papers/v13/bergstra12a.html">jmlr.org/papers/v13/bergstra12a.html</a> (JMLR assigns no DOI)</li>
<li id="r12">Snoek J, Larochelle H, Adams RP. Practical Bayesian optimization of machine learning algorithms. <em>arXiv</em>. 2012. <a href="https://doi.org/10.48550/arXiv.1206.2944">doi:10.48550/arXiv.1206.2944</a></li>
<li id="r13">Akiba T, Sano S, Yanase T, Ohta T, Koyama M. Optuna: a next-generation hyperparameter optimization framework. In: <em>Proc. 25th ACM SIGKDD</em>. ACM; 2019:2623-2631. <a href="https://doi.org/10.1145/3292500.3330701">doi:10.1145/3292500.3330701</a></li>
<li id="r14">Li L, Jamieson K, DeSalvo G, Rostamizadeh A, Talwalkar A. Hyperband: a novel bandit-based approach to hyperparameter optimization. <em>arXiv</em>. 2016. <a href="https://doi.org/10.48550/arXiv.1603.06560">doi:10.48550/arXiv.1603.06560</a></li>
<li id="r15">Varma S, Simon R. Bias in error estimation when using cross-validation for model selection. <em>BMC Bioinformatics</em>. 2006;7:91. <a href="https://doi.org/10.1186/1471-2105-7-91">doi:10.1186/1471-2105-7-91</a></li>
<li id="r16">Grinsztajn L, Oyallon E, Varoquaux G. Why do tree-based models still outperform deep learning on typical tabular data? <em>arXiv</em>. 2022. <a href="https://doi.org/10.48550/arXiv.2207.08815">doi:10.48550/arXiv.2207.08815</a></li>
<li id="r17">Shwartz-Ziv R, Armon A. Tabular data: deep learning is not all you need. <em>Inf Fusion</em>. 2022;81:84-90. <a href="https://doi.org/10.1016/j.inffus.2021.11.011">doi:10.1016/j.inffus.2021.11.011</a></li>
<li id="r18">van den Goorbergh R, van Smeden M, Timmerman D, Van Calster B. The harm of class imbalance corrections for risk prediction models: illustration and simulation using logistic regression. <em>J Am Med Inform Assoc</em>. 2022;29(9):1525-1534. <a href="https://doi.org/10.1093/jamia/ocac093">doi:10.1093/jamia/ocac093</a></li>
<li id="r19">Potharst R, Feelders AJ. Classification trees for problems with monotonicity constraints. <em>ACM SIGKDD Explor Newsl</em>. 2002;4(1):1-10. <a href="https://doi.org/10.1145/568574.568577">doi:10.1145/568574.568577</a></li>
<li id="r20">Lundberg SM, Erion G, Chen H, et al. From local explanations to global understanding with explainable AI for trees. <em>Nat Mach Intell</em>. 2020;2:56-67. <a href="https://doi.org/10.1038/s42256-019-0138-9">doi:10.1038/s42256-019-0138-9</a></li>
<li id="r21">Sheridan RP, Wang WM, Liaw A, Ma J, Gifford EM. Extreme gradient boosting as a method for quantitative structure-activity relationships. <em>J Chem Inf Model</em>. 2016;56(12):2353-2360. <a href="https://doi.org/10.1021/acs.jcim.6b00591">doi:10.1021/acs.jcim.6b00591</a></li>
<li id="r22">Probst P, Wright MN, Boulesteix AL. Hyperparameters and tuning strategies for random forest. <em>WIREs Data Min Knowl Discov</em>. 2019;9(3):e1301. <a href="https://doi.org/10.1002/widm.1301">doi:10.1002/widm.1301</a></li>
<li id="r23">Dietterich TG. Ensemble methods in machine learning. In: <em>Multiple Classifier Systems</em>. LNCS 1857. Springer; 2000:1-15. <a href="https://doi.org/10.1007/3-540-45014-9_1">doi:10.1007/3-540-45014-9_1</a></li>
<li id="r24">Wolpert DH. Stacked generalization. <em>Neural Netw</em>. 1992;5(2):241-259. <a href="https://doi.org/10.1016/S0893-6080(05)80023-1">doi:10.1016/S0893-6080(05)80023-1</a></li>
<li id="r25">Chawla NV, Bowyer KW, Hall LO, Kegelmeyer WP. SMOTE: synthetic minority over-sampling technique. <em>J Artif Intell Res</em>. 2002;16:321-357. <a href="https://doi.org/10.1613/jair.953">doi:10.1613/jair.953</a></li>
<li id="r26">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r27">Sheridan RP. Time-split cross-validation as a method for estimating the goodness of prospective prediction. <em>J Chem Inf Model</em>. 2013;53(4):783-790. <a href="https://doi.org/10.1021/ci400084k">doi:10.1021/ci400084k</a></li>
<li id="r28">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <em>J Med Chem</em>. 1996;39(15):2887-2893. <a href="https://doi.org/10.1021/jm9602928">doi:10.1021/jm9602928</a></li>
<li id="r29">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r30">Collins GS, Moons KGM, Dhiman P, et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. <em>BMJ</em>. 2024;385:e078378. <a href="https://doi.org/10.1136/bmj-2023-078378">doi:10.1136/bmj-2023-078378</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009 (free PDF). — Chapters 10 and 15: boosting and random forests, with the bias–variance analysis.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021 (free PDF). — Chapter 8, trees to boosting, with R code.</li>
<li>Kuhn M, Johnson K. <em>Applied Predictive Modeling</em>. Springer; 2013. — Resampling, tuning and model comparison in R.</li>
<li>Molnar C. <em>Interpretable Machine Learning</em>. 2nd ed.; 2022 (free online). — Partial dependence, monotonic constraints and what an ensemble can be made to say.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>XGBoost documentation, "Notes on parameter tuning" and the R package vignettes: <a href="https://xgboost.readthedocs.io/">xgboost.readthedocs.io</a>.</li>
<li><code>ranger</code> reference manual and the <code>caret</code> "Model Training and Tuning" chapter: <a href="https://topepo.github.io/caret/">topepo.github.io/caret</a>.</li>
<li>Optuna documentation (the Python tool this unit replaces by a hand-written surrogate search): <a href="https://optuna.readthedocs.io/">optuna.readthedocs.io</a>.</li>
<li>Practical Cheminformatics (P. Walters) — posts on gradient boosting for QSAR and on how little tuning usually matters: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
</ul>
